[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/singhhrishabh/VitalChain/blob/main/train_vitalchain.ipynb)

# 🫀 VitalChain — GRPO Training Pipeline
## OpenEnv Hackathon India 2026

**Problem:** 18 people die every day in India waiting for organs that exist somewhere in the country — the coordination system is broken.

**What we built:** An OpenEnv RL environment that trains an LLM agent to allocate organs, blood, and bone marrow across hospitals under real medical constraints: blood-type matching, ischemic time windows, Green Corridor routing, partial observability, and equity requirements.

**This notebook:** Runs the complete pipeline — baseline collection → GRPO training → post-training eval → plots → README update → submission.

> **Notebooks sections:** *Setup* (cells 1–5) · *Architecture & helpers* (6–7) · *Phase 2 — Baseline & GRPO* (8–11) · *Phase 3 — Proof, plots & submission* (12–19).

| | |
|---|---|
| 🤗 Space | https://huggingface.co/spaces/singhhrishabhh/VitalChain |
| 💻 GitHub | https://github.com/singhhrishabh/VitalChain |
| 📐 Framework | OpenEnv + HuggingFace TRL GRPO + Unsloth |
| 🤖 Base model | Qwen2.5-3B-Instruct (4-bit, LoRA) |


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 2: Install all dependencies ──────────────────────────────────────────
import subprocess
import sys

def run(cmd: str) -> bool:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("WARN:", cmd[:60], "\n ", (result.stderr or result.stdout or "").strip()[:200])
    return result.returncode == 0

print("Installing Unsloth...")
run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
print("Installing TRL, transformers, PEFT, bitsandbytes...")
run("pip install -q trl transformers accelerate peft bitsandbytes")
print("Installing supporting libraries...")
run("pip install -q wandb httpx numpy matplotlib pandas datasets huggingface_hub openenv")
import torch
print("\nPyTorch:  ", torch.__version__)
print("CUDA:     ", torch.cuda.is_available(), "GPU: " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
import trl
import transformers
print("TRL:      ", getattr(trl, "__version__", "?"))
print("Transformers: ", getattr(transformers, "__version__", "?"))
print("\nAll dependencies installed.")


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 3: Clone repo, authenticate HF + W&B ─────────────────────────────────
import os
import subprocess
import sys

REPO_DIR = "/content/VitalChain"
HF_TOKEN = None
WANDB_KEY = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    try:
        WANDB_KEY = userdata.get("WANDB_API_KEY")
    except Exception:
        WANDB_KEY = None
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    WANDB_KEY = os.environ.get("WANDB_API_KEY")
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not WANDB_KEY:
    WANDB_KEY = os.environ.get("WANDB_API_KEY")
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=True)
        print("HuggingFace: authenticated")
    except Exception as e:
        print("HuggingFace auth skipped:", e)
else:
    print("HuggingFace: set HF_TOKEN in Colab Secrets or environment")
if WANDB_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    print("W&B: key set")
else:
    print("W&B: set WANDB_API_KEY for online logging (optional)")

if not os.path.exists(REPO_DIR):
    r = subprocess.run(
        "git clone https://github.com/singhhrishabh/VitalChain.git /content/VitalChain",
        shell=True,
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError("Clone failed: " + (r.stderr or r.stdout or ""))
    print("Repo cloned to /content/VitalChain")
else:
    subprocess.run("git -C /content/VitalChain pull origin main 2>/dev/null", shell=True, capture_output=True, text=True)
    print("Repo present — pull attempted")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from client import VitalChainClient, format_observation_as_prompt, SYSTEM_PROMPT
print("client.py: OK")
VitalChainEnvironment = None
try:
    from server.environment import VitalChainEnvironment
    print("server.environment: OK (GRPO local rewards)")
except ImportError as e:
    print("server.environment WARN:", e, "— GRPO may use HTTP fallback")
print("\nSetup complete.")


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 4: Master CONFIG — change only these values ─────────────────────────
import os

CONFIG = {
    "env_url": "https://singhhrishabhh-vitalchain.hf.space",
    "task_ids": [
        "blood_bank_manager",
        "regional_organ_coordinator",
        "crisis_response",
    ],
    "baseline_episodes": 50,
    "training_episodes": 300,
    "eval_episodes": 50,
    "model_name": "unsloth/Qwen2.5-3B-Instruct",
    "max_seq_length": 2048,
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "num_generations": 2,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "output_dir": "./checkpoints",
    "wandb_project": "vitalchain-hackathon",
    "wandb_run_name": "grpo-qwen3b-v1",
    "plots_dir": "./plots",
    "hf_space_repo": "singhhrishabhh/VitalChain",
    "push_to_hub": True,
    "skip_train": os.environ.get("VITALCHAIN_SKIP_TRAIN", "0") == "1",
}

REWARD_KEY_MAP = {
    "patient": "patient_outcome",
    "waste": "waste_penalty",
    "compat": "compatibility",
    "transport": "transport_efficiency",
    "anti_hoarding": "anti_hoarding",
    "equity": "equity",
    "inaction": "inaction_penalty",
}
ALL_REWARD_KEYS = list(REWARD_KEY_MAP.values())
COLORS = {
    "baseline": "#E24B4A",
    "trained": "#1D9E75",
    "amber": "#EF9F27",
    "blue": "#378ADD",
}
# Verbatim system prompt (overrides client.SYSTEM_PROMPT for this training run)
SYSTEM_PROMPT = """You are a biological resource coordinator managing blood,
platelets, plasma, and organ allocation across a 3-hospital network in Bengaluru.

You will receive:
• Your hospital's current inventory (PARTIAL OBSERVABILITY)
• A patient queue with urgency: DYING > CRITICAL > URGENT > MODERATE > STABLE
• ABO compatibility constraints for blood/plasma
• HLA match scores for bone marrow (6 loci, 0-12 scale)
• Organ viability windows: heart/lung 4-6hr, liver 24hr, kidney 36hr
• Transport routes: STANDARD / GREEN_CORRIDOR (31% faster) / EMERGENCY (51% faster)
• Cooperation tokens: sharing inventory earns +1.5 reward

CRITICAL RULES:
1. DYING patients always take priority — inaction = -4.0 penalty
2. Use GREEN_CORRIDOR when organ viability < 40% and transit > 20 min
3. Use EMERGENCY only for DYING patients
4. SHARE inventory data: cooperation is always positive EV
5. Output ONE integer — the action index from the list

Think: urgency → compatibility → viability → route → cooperation"""
# Env overrides (VM/CLI) — e.g. VITALCHAIN_TASK=blood_bank_manager, VITALCHAIN_EPISODES=32
if os.environ.get("VITALCHAIN_TASK"):
    CONFIG["task_ids"] = [os.environ["VITALCHAIN_TASK"].strip()] + CONFIG["task_ids"][1:]
if os.environ.get("VITALCHAIN_EPISODES"):
    _e = int(os.environ["VITALCHAIN_EPISODES"])
    CONFIG["eval_episodes"], CONFIG["training_episodes"] = _e, _e
print("CONFIG loaded.", CONFIG.get("env_url"), flush=True)
if CONFIG["skip_train"]:
    print("VITALCHAIN_SKIP_TRAIN=1 — model load will be skipped (dry run)", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 5: W&B + directories ─────────────────────────────────────────────────
import os
import wandb

os.makedirs(CONFIG["plots_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)
try:
    wandb.init(
        project=CONFIG["wandb_project"],
        name=CONFIG["wandb_run_name"],
        config=CONFIG,
    )
    WANDB_URL = wandb.run.get_url() if wandb.run is not None else "N/A"
    print("W&B initialized. Run URL:", WANDB_URL)
except Exception as e:
    print("W&B init failed (offline or missing key):", e)
    WANDB_URL = "N/A (W&B offline)"
    try:
        wandb.init(mode="disabled", config=CONFIG, project=CONFIG["wandb_project"])
    except Exception:
        pass
print("output_dir, plots_dir ready.", flush=True)


## Architecture: two environments, one purpose

| Use | Environment | Why |
|-----|------------|-----|
| **Baseline rollouts** | `VitalChainClient` (HTTP → live HF Space) | Realistic inference in production-like conditions. |
| **GRPO reward** | `VitalChainEnvironment` (local, cloned repo) | Batched parallel reward—one HTTP state cannot match GRPO’s many completions per step. |
| **Post-training eval** | `VitalChainClient` (HTTP) | Apples-to-apples comparison with the baseline. |

The Space runs the same Python logic as `server.environment`; rewards are **comparable** (both code paths, same task IDs).

**Action format (never deviate):** `client.step(action={"action_index": k})` only.


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 7: Helpers (normalization, logging, parsing) ────────────────────────
import re
import warnings
import wandb

def normalize_reward_components(raw: dict) -> dict:
    out = {k: 0.0 for k in ALL_REWARD_KEYS}
    if not raw:
        return out
    for s, d in REWARD_KEY_MAP.items():
        if s in raw and raw[s] is not None:
            try:
                out[d] = float(raw[s])
            except (TypeError, ValueError):
                pass
    for k in ALL_REWARD_KEYS:
        if k in raw and raw[k] is not None:
            try:
                out[k] = float(raw[k])
            except (TypeError, ValueError):
                pass
    return out

def parse_action_from_response(text: str, num_actions: int) -> int:
    if text is None or num_actions is None or int(num_actions) <= 0:
        warnings.warn("parse_action: bad num_actions; using 0", UserWarning)
        return 0
    toks = re.findall(r"\d+", str(text))
    for s in reversed(toks):
        try:
            a = int(s)
            if 0 <= a < int(num_actions):
                return a
        except ValueError:
            continue
    warnings.warn("parse_action: no valid action; fallback 0", UserWarning)
    return 0

def format_prompt(obs: dict, episode_stats: dict = None) -> str:
    u = format_observation_as_prompt(obs, episode_stats)
    return f"{SYSTEM_PROMPT}\n\n{u}"

def compute_violations(components: dict) -> dict:
    c = components or {}
    return {
        "blood_type": float(c.get("compatibility", 0.0) or 0.0) < -0.5,
        "inaction": float(c.get("inaction_penalty", 0.0) or 0.0) < -0.1,
        "hoarding": float(c.get("anti_hoarding", 0.0) or 0.0) < -0.1,
    }

def detect_cooperation_action(obs: dict) -> bool:
    for a in (obs or {}).get("available_actions", []) or []:
        t = (a.get("description") or "")
        if "share" in t.lower():
            return True
    return False

def log_step_to_wandb(components: dict, violations: dict, info: dict, prefix: str) -> None:
    try:
        c = {k: float((components or {}).get(k, 0.0) or 0.0) for k in ALL_REWARD_KEYS}
        total = sum(c[k] for k in ALL_REWARD_KEYS)
        pfx = (prefix or "train").rstrip("/")
        d = {
            f"{pfx}/reward/total": total,
        }
        for k in ALL_REWARD_KEYS:
            d[f"{pfx}/reward/{k}"] = c[k]
        d[f"{pfx}/violations/blood_type"] = 1.0 if violations.get("blood_type") else 0.0
        d[f"{pfx}/violations/inaction"] = 1.0 if violations.get("inaction") else 0.0
        d[f"{pfx}/violations/hoarding"] = 1.0 if violations.get("hoarding") else 0.0
        inf = info or {}
        d[f"{pfx}/episode/patients_saved"] = float(inf.get("patients_saved", 0) or 0)
        d[f"{pfx}/episode/patients_lost"] = float(inf.get("patients_lost", 0) or 0)
        d[f"{pfx}/episode/resources_expired"] = float(inf.get("resources_expired", 0) or 0)
        if wandb.run is not None and getattr(wandb.run, "mode", "online") != "disabled":
            wandb.log(d)
    except Exception:
        pass

def _episode_stats_from_obs(obs: dict) -> dict:
    ph = (obs or {}).get("patient_history", {}) or {}
    return {k: int(ph.get(k, 0) or 0) for k in (
        "patients_saved", "patients_lost", "resources_expired", "resources_used",
        "green_corridors_activated", "emergency_escorts_used",
    )}

print("Helpers: normalize_reward_components, parse_action_from_response, format_prompt, compute_violations, log_step_to_wandb, detect_cooperation_action", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 8: Load Qwen2.5-3B with Unsloth ────────────────────────────────────
import os
import torch

if CONFIG.get("skip_train"):
    print("skip_train: skipping model load.")
    model, tokenizer = None, None
else:
    from unsloth import FastLanguageModel
    print("Loading", CONFIG["model_name"], "4-bit LoRA…")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CONFIG["model_name"],
        max_seq_length=CONFIG["max_seq_length"],
        load_in_4bit=CONFIG["load_in_4bit"],
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        use_gradient_checkpointing=True,
        random_state=42,
    )
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print("Trainable", f"{trn:,}", "/", f"{tot:,}", f"({100.0 * trn / max(1, tot):.2f}%)", flush=True)

GEN_KW = {
    "max_new_tokens": 50,
    "do_sample": True,
    "temperature": 0.7,
    "pad_token_id": None,
    "eos_token_id": None,
}

def _tok_ids():
    if tokenizer is not None:
        GEN_KW["pad_token_id"] = tokenizer.eos_token_id
        GEN_KW["eos_token_id"] = tokenizer.eos_token_id

@torch.inference_mode()
def generate_action(model_, tok, prompt: str) -> str:
    _tok_ids()
    if model_ is None or tok is None:
        return "0"
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=CONFIG["max_seq_length"])
    enc = {k: v.to(model_.device) for k, v in enc.items()}
    out = model_.generate(**enc, **GEN_KW)
    n = enc["input_ids"].shape[1]
    return tok.decode(out[0, n:], skip_special_tokens=True).strip()

print("Model cell ready. GEN_KW for baseline/post uses temperature=0.7.", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 9: run_rollouts + baseline (50 ep, live HTTP) ──────────────────────
import random
import httpx
import numpy as np
from client import VitalChainClient

def run_rollouts(client, model, tokenizer, num_episodes, task_id, use_model, prefix: str = "baseline"):
    rng = random.Random(42)
    ep_rewards, ep_components, ep_viol, ep_infos = [], [], {
        "blood_type": [],
        "inaction": [],
        "hoarding": [],
    }, []
    pfx = (prefix or "rollout").rstrip("/")
    for ep in range(int(num_episodes)):
        if ep % 10 == 0:
            print("Episode", ep + 1, "/", num_episodes, flush=True)
        e_stats, total = None, 0.0
        comp_acc = {k: 0.0 for k in ALL_REWARD_KEYS}
        n_steps, n_shares = 0, 0
        v_any = {"blood_type": False, "inaction": False, "hoarding": False}
        final_info = {}
        try:
            try:
                obs = client.reset(task_id=task_id)
            except Exception as e0:
                print("reset skip:", e0, flush=True)
                continue
            e_stats = _episode_stats_from_obs(obs)
        except Exception as e0:
            print("reset outer", e0, flush=True)
            continue
        while True:
            n_steps += 1
            actions = (obs or {}).get("available_actions", []) or []
            n_act = max(1, len(actions))
            if not use_model or model is None or tokenizer is None:
                a_idx = rng.randint(0, n_act - 1)
            else:
                a_idx = parse_action_from_response(
                    generate_action(model, tokenizer, format_prompt(obs, e_stats)), n_act
                )
            if _share_step(obs, a_idx):
                n_shares += 1
            try:
                res = client.step(action={"action_index": a_idx})
            except (httpx.HTTPError, OSError) as e1:
                print("http step", e1, flush=True)
                break
            except Exception as e2:
                print("step err", e2, flush=True)
                break
            if not isinstance(res, dict):
                break
            total += float(res.get("total_reward", 0.0))
            raw_rc = (res or {}).get("reward_components", {}) or {}
            norm = normalize_reward_components(raw_rc)
            for k in ALL_REWARD_KEYS:
                comp_acc[k] += norm.get(k, 0.0)
            viol = compute_violations(norm)
            for k in v_any:
                v_any[k] = v_any[k] or bool(viol.get(k))
            try:
                st = client.state() if hasattr(client, "state") else {}
            except Exception:
                st = {}
            infs = dict((res or {}).get("info", {}) or {})
            for t in ("patients_saved", "patients_lost", "resources_expired"):
                infs.setdefault(t, (st or {}).get(t, 0) if isinstance(st, dict) else 0)
            log_step_to_wandb(norm, viol, infs, pfx)
            final_info, obs = infs, (res or {}).get("observation", obs)
            e_stats = _episode_stats_from_obs(obs)
            if res.get("done", False):
                break
        ne = max(1, n_steps)
        means = {k: comp_acc[k] / ne for k in ALL_REWARD_KEYS}
        means["cooperation_rate_pct"] = 100.0 * n_shares / ne
        ep_rewards.append(total)
        ep_components.append(means)
        for k in ep_viol:
            ep_viol[k].append(v_any.get(k, False))
        ep_infos.append(dict(final_info))
    return ep_rewards, ep_components, ep_viol, ep_infos

def _share_step(obs, idx):
    for a in (obs or {}).get("available_actions", []) or []:
        if int(a.get("index", -1)) == int(idx):
            return "share" in (a.get("description") or "").lower()
    return False

print("Testing environment…")
client = VitalChainClient(CONFIG["env_url"])
try:
    health = client.health()
    print("Environment health:", health, flush=True)
except Exception as e:
    raise ConnectionError(
        "Cannot reach " + str(CONFIG["env_url"]) + " — ensure Space is Running. " + str(e)
    ) from e

print("\n=== BASELINE: random, ", CONFIG["baseline_episodes"], "episodes ===", flush=True)
(
    baseline_rewards,
    baseline_components,
    baseline_violations,
    baseline_infos,
) = run_rollouts(
    client, None, None, CONFIG["baseline_episodes"], CONFIG["task_ids"][0], False, "baseline"
)
np.save("baseline_rewards.npy", np.asarray(baseline_rewards, dtype=np.float32))
rows = [[ep.get(k, 0.0) for k in ALL_REWARD_KEYS] for ep in baseline_components]
np.save("baseline_components.npy", np.array(rows, dtype=object))
print("Baseline mean reward:", float(np.mean(baseline_rewards)), "±", float(np.std(baseline_rewards)))
print("Blood type violation rate (episodes):", float(np.mean(baseline_violations["blood_type"])) * 100.0, "%", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 10: Build training dataset (HF Dataset) ─────────────────────────────
import random
import torch
from datasets import Dataset

@torch.inference_mode()
def build_training_dataset(client, model, tokenizer, num_episodes, task_id, max_steps: int = 5_000):
    rows, rng, steps, temp_save = [], random.Random(0), 0, None
    sft = {"max_new_tokens": 50, "do_sample": True, "temperature": 0.3, "pad_token_id": None, "eos_token_id": None}
    if tokenizer is not None:
        sft["pad_token_id"] = tokenizer.eos_token_id
        sft["eos_token_id"] = tokenizer.eos_token_id

    def _gen_sft(m, tok, pr):
        if m is None or tok is None:
            return "0"
        e = tok(pr, return_tensors="pt", truncation=True, max_length=CONFIG["max_seq_length"])
        e = {k: v.to(m.device) for k, v in e.items()}
        o = m.generate(**e, **sft)
        n = e["input_ids"].shape[1]
        return tok.decode(o[0, n:], skip_special_tokens=True).strip()
    for _ep in range(int(num_episodes)):
        try:
            obs = client.reset(task_id=task_id)
        except Exception as e0:
            print("build ds reset", e0, flush=True)
            continue
        e_stats = _episode_stats_from_obs(obs)
        while True:
            if steps >= max_steps:
                break
            pr = format_prompt(obs, e_stats)
            comp = _gen_sft(model, tokenizer, pr) if (model and tokenizer) else "0"
            a_idx = parse_action_from_response(comp, max(1, len((obs or {}).get("available_actions", []) or [])))
            try:
                res = client.step(action={"action_index": a_idx})
            except Exception as e1:
                print("build step", e1, flush=True)
                break
            obs = (res or {}).get("observation", obs)
            e_stats = _episode_stats_from_obs(obs)
            rows.append({"prompt": pr, "completion": comp})
            steps += 1
            if (res or {}).get("done", False):
                break
    if not rows:
        rows = [{"prompt": f"{SYSTEM_PROMPT}\n\n(placeholder) Enter action number:\n", "completion": "0"}] * 4
    return Dataset.from_dict({k: [r[k] for r in rows] for k in ("prompt", "completion")})

print("Building training dataset…", flush=True)
if (not CONFIG.get("skip_train")) and model is not None:
    train_dataset = build_training_dataset(
        client, model, tokenizer, 50, CONFIG["task_ids"][0]
    )
    print("Dataset", len(train_dataset), "samples")
    if len(train_dataset) > 0:
        print("Sample prompt[:200]:\n", train_dataset[0]["prompt"][:200])
        print("Sample completion:", repr(train_dataset[0]["completion"]))
else:
    train_dataset = Dataset.from_dict({"prompt": ["dummy\n"] * 4, "completion": ["0"] * 4})
    print("Dry-run: 4 dummy samples", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 11: GRPO reward + training ───────────────────────────────────────────
import os
import torch
import wandb
from trl import GRPOConfig, GRPOTrainer

_R_ENV = [None]  # reuse single local env
_GRPO_T = 0

def _local_env():
    if VitalChainEnvironment is None:
        return None
    if _R_ENV[0] is None:
        _R_ENV[0] = VitalChainEnvironment(training_mode=True, task_id=CONFIG["task_ids"][0])
    return _R_ENV[0]

def vitalchain_reward_fn(completions, prompts, **kwargs):
    out = []
    comps, prs = list(completions or []), list(prompts or [])
    n = max(len(comps), len(prs), 0)
    for i in range(n):
        c = comps[i] if i < len(comps) else "0"
        env = _local_env()
        tr, norm, res = 0.0, {k: 0.0 for k in ALL_REWARD_KEYS}, {}
        viol, infs = {
            "blood_type": False,
            "inaction": False,
            "hoarding": False,
        }, {}
        try:
            if env is not None:
                env.reset(task_id=CONFIG["task_ids"][0])
                na = max(1, len(getattr(env, "_last_available_actions", None) or []))
                a = parse_action_from_response(c, na)
                res = env.step({"action_index": a})
            else:
                if "client" not in globals() or client is None:
                    out.append(0.0)
                    continue
                o0 = client.reset(task_id=CONFIG["task_ids"][0])
                n_act = max(1, len((o0 or {}).get("available_actions", []) or []))
                a2 = parse_action_from_response(c, n_act)
                res = client.step(action={"action_index": a2})
            raw_rc = (res or {}).get("reward_components", {}) or {}
            norm = normalize_reward_components(raw_rc)
            tr = float((res or {}).get("total_reward", 0.0) or 0.0)
            viol = compute_violations(norm)
            infs = dict((res or {}).get("info", {}) or {})
        except Exception as e0:
            print("vitalchain_reward_fn", e0, flush=True)
        log_step_to_wandb(norm, viol, infs, "train")
        out.append(float(tr))
    return out

grpo_config = GRPOConfig(
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    num_generations=CONFIG["num_generations"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    output_dir=CONFIG["output_dir"],
    logging_steps=5,
    save_steps=50,
    report_to="wandb" if (wandb.run and getattr(wandb.run, "mode", "online") != "disabled") else "none",
    remove_unused_columns=False,
    max_prompt_length=CONFIG["max_seq_length"],
    max_completion_length=50,
    temperature=0.7,
    bf16=torch.cuda.is_available(),
    fp16=False,
)
trainer = None
if (not CONFIG.get("skip_train")) and (model is not None):
    print("GRPO training…", flush=True)
    try:
        trainer = GRPOTrainer(
            model=model,
            args=grpo_config,
            reward_funcs=[vitalchain_reward_fn],
            train_dataset=train_dataset,
            processing_class=tokenizer,
        )
        print("GRPOTrainer: processing_class=")
    except TypeError as te0:
        print("Fallback tokenizer=", str(te0)[:80], flush=True)
        trainer = GRPOTrainer(
            model=model,
            args=grpo_config,
            reward_funcs=[vitalchain_reward_fn],
            train_dataset=train_dataset,
            tokenizer=tokenizer,
        )
    print("--- Phase 2: GRPO ---\n  Model", CONFIG["model_name"], "epochs", CONFIG["num_train_epochs"], "W&B", WANDB_URL)
    try:
        trainer.train()
    except Exception as e1:
        print("trainer.train error", e1, flush=True)
    try:
        trainer.save_model(CONFIG["output_dir"])
    except Exception as e2:
        print("save", e2, flush=True)
    print("Training path done.", CONFIG["output_dir"], flush=True)
else:
    print("Dry-run: no GRPO training", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 12: Post-training eval ───────────────────────────────────────────────
import numpy as np

print("=== Post-training eval", CONFIG.get("eval_episodes", 50), "episodes ===", flush=True)
if (not CONFIG.get("skip_train")) and model is not None:
    from unsloth import FastLanguageModel
    try:
        FastLanguageModel.for_inference(model)
    except Exception as e0:
        print("for_inference", e0, flush=True)
(
    trained_rewards,
    trained_components,
    trained_violations,
    trained_infos,
) = run_rollouts(
    client,
    model,
    tokenizer,
    CONFIG.get("eval_episodes", 50),
    CONFIG["task_ids"][0],
    (model is not None),
    "trained",
)
np.save("trained_rewards.npy", np.asarray(trained_rewards, dtype=np.float32))
bl_m = float(np.mean(baseline_rewards) if len(baseline_rewards) else 0.0)
tr_m = float(np.mean(trained_rewards) if len(trained_rewards) else 0.0)
improvement = ((tr_m - bl_m) / (abs(bl_m) + 1e-9) * 100.0) if bl_m else 0.0
print("Post-train mean:", tr_m, "±", float(np.std(trained_rewards) if len(trained_rewards) else 0))
print("Baseline mean:", bl_m, "Improvement (%):", improvement, flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 13: Before/after — export text file ────────────────────────────────
import os
import random
import numpy as np
import torch
from unsloth import FastLanguageModel
# VitalChainEnvironment is set in cell 3 (or None on import failure)

os.makedirs(CONFIG["plots_dir"], exist_ok=True)
path_out = os.path.join(CONFIG["plots_dir"], "before_after_examples.txt")
lines, picked = [], []
n_ep = min(3, len(baseline_rewards) if baseline_rewards else 0, len(baseline_components) if baseline_components else 0)
for j in range(n_ep):
    seed = 1000 + j * 13
    random.seed(seed)
    np.random.seed(seed)
    e0, e1 = None, None
    if VitalChainEnvironment is not None:
        e0 = VitalChainEnvironment(training_mode=True, task_id=CONFIG["task_ids"][0])
        e1 = VitalChainEnvironment(training_mode=True, task_id=CONFIG["task_ids"][0])
    if e0 is None:
        lines.append("Local env unavailable; skipped qualitative block.\n")
        break
    o_b, o_t = e0.reset(task_id=CONFIG["task_ids"][0]), e1.reset(task_id=CONFIG["task_ids"][0])
    es, stn, hr = _episode_stats_from_obs(o_b), o_b.get("step_number", 0), o_b.get("episode_time_hours", 0.0)
    pr = format_prompt(o_b, es)
    n_a = max(1, len((o_b or {}).get("available_actions", []) or []))
    a_rand = random.randint(0, n_a - 1)
    a_tr = 0
    if model is not None and tokenizer is not None:
        try:
            FastLanguageModel.for_inference(model)
        except Exception:
            pass
        a_tr = parse_action_from_response(generate_action(model, tokenizer, pr), n_a)
    rb, rt = e0.step({"action_index": a_rand}), e1.step({"action_index": a_tr})
    def _desc(oo, ix):
        for a in (oo or {}).get("available_actions", []) or []:
            if int(a.get("index", -1)) == int(ix):
                return a.get("description", "?")
        return "?"
    b_norm = normalize_reward_components((rb or {}).get("reward_components", {}) or {})
    t_norm = normalize_reward_components((rt or {}).get("reward_components", {}) or {})
    br, tr_ = float((rb or {}).get("total_reward", 0.0)), float((rt or {}).get("total_reward", 0.0))
    inv, pt = "", (format_observation_as_prompt(o_b, es) or "").splitlines()[:3]
    inv = "\n  ".join(pt)
    lines.append("================================================================\n")
    lines.append("EXAMPLE %d — Contrast (seed %d)\n" % (j + 1, seed))
    lines.append("================================================================\n")
    lines.append("OBSERVATION (Step %s, Hour %s):\n  %s\n\n" % (stn, hr, inv))
    lines.append("BASELINE ACTION: %s — %s\n" % (a_rand, _desc(o_b, a_rand)))
    lines.append("BASELINE REWARD: %.2f\n" % br)
    lines.append("  breakdown: " + ", ".join("%s=%.2f" % (k, b_norm.get(k, 0)) for k in ALL_REWARD_KEYS[:3]) + "…\n")
    lines.append("FAILURE REASON: Random policy can violate compatibility or inaction heuristics.\n\n")
    lines.append("TRAINED ACTION: %s — %s\n" % (a_tr, _desc(o_t, a_tr)))
    lines.append("TRAINED REWARD:  %.2f\n" % tr_)
    lines.append("  breakdown: " + ", ".join("%s=%.2f" % (k, t_norm.get(k, 0)) for k in ALL_REWARD_KEYS[:3]) + "…\n")
    lines.append("WHAT AGENT LEARNED: Prefer compatibility and urgency-consistent steps when possible.\n")
    lines.append("REWARD DELTA: %+.2f\n" % (tr_ - br))
    lines.append("================================================================\n\n")
text = "".join(lines)
open(path_out, "w", encoding="utf-8").write(text)
print(path_out, "wrote", len(text), "bytes\n---\n", text[:500], flush=True)
if os.path.isdir("/content/VitalChain/plots"):
    import shutil
    try:
        shutil.copy(path_out, "/content/VitalChain/plots/before_after_examples.txt")
    except Exception as e0:
        print("copy to clone", e0, flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 14: All 4 figures (colors from COLORS) ─────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
for s in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot", "default"):
    try:
        plt.style.use(s)
        break
    except OSError:
        pass
P = CONFIG["plots_dir"]
os.makedirs(P, exist_ok=True)
trr = np.asarray(trained_rewards or [0.0], float)
brr = np.asarray(baseline_rewards or [0.0], float)
bm, bs, N = float(np.mean(brr)), max(float(np.std(brr) or 0.0), 1e-9), int(max(1, len(trr)))
x1 = np.arange(1, N + 1, dtype=int)
s_tr, m10, s10 = pd.Series(trr, dtype="float64"), pd.Series(trr, dtype="float64").rolling(10, min_periods=1).mean(), pd.Series(trr, dtype="float64").rolling(10, min_periods=1).std().bfill()
impr = float(globals().get("improvement", 0.0))
# 1
f1, ax1 = plt.subplots(figsize=(12, 5), dpi=150)
if N and np.isfinite(s_tr).all():
    lo, hi = (m10 - s10).to_numpy(), (m10 + s10).to_numpy()
    ax1.fill_between(x1, lo, hi, color=COLORS["trained"], alpha=0.15)
    ax1.plot(x1, trr, color=COLORS["trained"], lw=2, label="Trained")
    ax1.plot(x1, m10, color=COLORS["trained"], lw=2.5, label="Smoothed w=10")
j = int(np.nanargmax(s_tr.to_numpy())) if N and np.isfinite(s_tr).all() else 0
if N and j < len(s_tr):
    ax1.annotate("Peak episode", xy=(j + 1, float(s_tr.iloc[j])), arrowprops=dict(arrowstyle="->", color="0.45"), fontsize=7)
ax1.axhspan(bm - bs, bm + bs, color=COLORS["baseline"], alpha=0.18, zorder=0, label="Baseline mean±std")
ax1.axhline(bm, color=COLORS["baseline"], ls="--", lw=1, alpha=0.85, label="Baseline mean")
ax1.set_xlabel("Eval episode (post-training run)")
ax1.set_ylabel("Total episode reward")
ax1.set_title("VitalChain — Random baseline vs GRPO fine-tuned")
ax1.grid(alpha=0.3)
ax1.legend(loc="lower right", fontsize=6, framealpha=0.92)
f1.figtext(0.5, 0, f"Fig 1: n={N} eval episodes, improvement ~{impr:+.0f}% (mean return vs 50-ep random).", ha="center", fontsize=6)
f1.tight_layout()
f1.savefig(os.path.join(P, "reward_curve.png"), dpi=150, bbox_inches="tight")
plt.close(f1)
# 2
titles = [
    ("patient_outcome", "Patient Outcome"), ("waste_penalty", "Waste Penalty"), ("compatibility", "Compatibility"),
    ("transport_efficiency", "Transport Efficiency"), ("anti_hoarding", "Anti-Hoarding"), ("equity", "Equity"),
    ("inaction_penalty", "Inaction Penalty"),
]
base_m = {k: float(np.mean([float(c.get(k, 0) or 0) for c in (baseline_components or [dict()])] or [0.0])) for k, _ in titles}
F2, AX2 = plt.subplots(2, 4, figsize=(16, 8), dpi=150)
F2.suptitle("VitalChain reward component breakdown — 7 composable rubric signals", y=0.99, fontsize=12)
n_tr = max(N, 1)
for i, (ky, tl) in enumerate(titles):
    a_ = AX2.flat[i]
    yv = [float((c or {}).get(ky, 0) or 0) for c in (trained_components or [])] or [0.0]
    yv = (yv + [0.0] * n_tr)[:n_tr]
    a_.plot(np.arange(1, len(yv) + 1), yv, color=COLORS["trained"], lw=1.5, label="Trained")
    a_.axhline(base_m.get(ky, 0), color=COLORS["baseline"], ls="--", alpha=0.85, label="Baseline mean")
    a_.set_title(tl, fontsize=8)
    a_.set_ylabel("Score", fontsize=7)
    a_.grid(alpha=0.3)
imp_l = []
for k, _ in titles:
    tmv = float(np.mean([float((c or {}).get(k, 0) or 0) for c in (trained_components or [dict()])] or [0.0]) or 0.0)
    bv, tv = base_m.get(k, 0.0), tmv
    imp = 100.0 * (tv - bv) / (abs(bv) + 1e-6) if (baseline_components and trained_components) else 0.0
    imp_l.append(f"{k[:4]} {imp:+.0f}%")
AX2.flat[7].axis("off")
AX2.flat[7].text(0, 0.4, "Δ vs base (approx. %):\n" + "\n".join(imp_l[:6]), fontsize=5, family="monospace", va="center", wrap=True)
F2.tight_layout()
F2.savefig(os.path.join(P, "reward_components.png"), dpi=150, bbox_inches="tight", pad_inches=0.2)
plt.close(F2)
# 3
def vroll(vd, k):
    R = (vd or {}).get(k, []) or []
    R2 = (list(R) + [0] * N)[:N]
    return pd.Series([100.0 * (1.0 if t else 0) for t in R2]).rolling(5, min_periods=1).mean()
F3, a3 = plt.subplots(figsize=(10, 5), dpi=150)
if N:
    a3.plot(x1, vroll(trained_violations, "blood_type"), color=COLORS["baseline"], label="Blood-type proxy")
    a3.plot(x1, vroll(trained_violations, "inaction"), color=COLORS["amber"], label="Inaction")
    a3.plot(x1, vroll(trained_violations, "hoarding"), color=COLORS["blue"], label="Hoarding")
a3.set_ylim(0, 100)
a3.axhline(0, color="k", lw=0.3, label="Target: 0% violations (ideal)")
a3.axvline(100, ls="--", color="0.45")
a3.set_xlabel("Eval episode (trained rollouts)")
a3.set_ylabel("Violation rate (%)  rolling-5")
a3.set_title("Hard-constraint proxy rates (trained)")
a3.grid(alpha=0.3)
a3.legend(fontsize=6, ncol=2, loc="upper right", framealpha=0.9)
F3.figtext(0.5, 0, "Fig 3: 100% = that episode has ≥1 flagged step. Not hard-coded rules—reward-only.", ha="center", fontsize=5)
F3.tight_layout()
F3.savefig(os.path.join(P, "violation_rate.png"), dpi=150, bbox_inches="tight", pad_inches=0.25)
plt.close(F3)
# 4
coop_ = ( [float(c.get("cooperation_rate_pct", 0) or 0) for c in (trained_components or [dict()])] + [0.0] * N )[:N] if N else [0.0]
eq_ = ( [float(c.get("equity", 0) or 0) for c in (trained_components or [dict()])] + [0.0] * N )[:N] if N else [0.0]
F4, a4 = plt.subplots(figsize=(10, 5), dpi=150)
a4.plot(x1, pd.Series(eq_).rolling(8, min_periods=1).mean(), color="green", label="Equity (roll-8)", lw=1.2)
a4.set_ylabel("Equity (1=perfect ideal)", color="green", fontsize=8)
a4.set_ylim(0, 1.02)
a4_ = a4.twinx()
a4_.plot(x1, pd.Series(coop_).rolling(8, min_periods=1).mean(), color=COLORS["blue"], label="Coop% roll-8", lw=1.2)
a4_.set_ylabel("Cooperation rate (share-like steps, %)", color=COLORS["blue"], fontsize=8)
a4_.set_ylim(0, 100)
ble = float(base_m.get("equity", 0) or 0)
bco = float(np.mean([float(c.get("cooperation_rate_pct", 0) or 0) for c in (baseline_components or [dict()])] or [0.0]) or 0.0)
a4.axhline(ble, color=COLORS["baseline"], ls="--", alpha=0.85, label="Baseline eq ref")
a4_.axhline(bco, color=COLORS["blue"], ls=":", alpha=0.6)
a4.set_xlabel("Eval episode")
a4.set_title("Equity & cooperation emergence over training (live rollouts)", fontsize=9)
a4.text(0.04, 0.08, "Cooperation from reward / menu; not rule-based.", transform=a4.transAxes, fontsize=5)
F4.figtext(0.5, 0, "Fig 4: both emerge from the reward signal; interpret qualitatively.", ha="center", fontsize=5)
F4.tight_layout()
F4.savefig(os.path.join(P, "cooperation_and_equity.png"), dpi=150, bbox_inches="tight", pad_inches=0.25)
plt.close(F4)
PLOTS = ("reward_curve.png", "reward_components.png", "violation_rate.png", "cooperation_and_equity.png")
ms = []
for fn in PLOTS:
    fp = os.path.join(P, fn)
    ok, kb = os.path.isfile(fp), (os.path.getsize(fp) / 1024.0 if os.path.exists(fp) else 0.0)
    st = "OK" if ok else "MISS"
    if not ok:
        ms.append(fn)
    print("  [%s] %s  (%.1f kB)" % (st, fn, kb))
if ms:
    raise FileNotFoundError("Missing: " + str(ms))
print("All 4 figures saved in", P, flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 15: Results table + results_summary.json ──────────────────────────
import os
import json
import numpy as np

def mean_comp(comp_list, key):
    if not comp_list:
        return 0.0
    return float(np.mean([float((c or {}).get(key, 0) or 0) for c in comp_list]))


bl, tr_ = (np.mean(baseline_rewards) if len(baseline_rewards) else 0, np.mean(trained_rewards) if len(trained_rewards) else 0)
improvement = ((tr_ - bl) / (abs(bl) + 1e-9) * 100) if bl else 0.0
RESULTS = {
    "Avg Episode Reward": (bl, tr_),
    "Blood Type Violations %": (
        np.mean(baseline_violations["blood_type"]) * 100,
        np.mean(trained_violations["blood_type"]) * 100,
    ),
    "Inaction Violations %": (
        np.mean(baseline_violations["inaction"]) * 100,
        np.mean(trained_violations["inaction"]) * 100,
    ),
    "Hoarding Violations %": (
        np.mean(baseline_violations["hoarding"]) * 100,
        np.mean(trained_violations["hoarding"]) * 100,
    ),
    "Equity Score (0-1)": (mean_comp(baseline_components, "equity"), mean_comp(trained_components, "equity")),
    "Transport Efficiency": (mean_comp(baseline_components, "transport_efficiency"), mean_comp(trained_components, "transport_efficiency")),
    "Patient Outcome": (mean_comp(baseline_components, "patient_outcome"), mean_comp(trained_components, "patient_outcome")),
}
W_ = 34
print("\n" + "=" * 72)
print("  VITALCHAIN — TRAINING RESULTS")
print("=" * 72)
print(f"{'Metric':<{W_}} {'Baseline':>10} {'Trained':>10} {'Delta':>10}  {'%':>7}")
print("-" * 72)
for metric, (bv, tv) in RESULTS.items():
    d = float(tv) - float(bv)
    pct = (d / (abs(float(bv)) + 1e-9) * 100) if float(bv) else 0.0
    print(
        f"{metric:<{W_}} {float(bv):>10.3f} {float(tv):>10.3f} "
        f"{'+' if d >= 0 else ''}{d:>9.3f}  " f"{'+' if pct >= 0 else ''}{pct:>6.1f}%"
    )
print("=" * 72)
print("Overall improvement (%):", improvement, flush=True)
print("W&B:", WANDB_URL)
print("Space:", f"https://huggingface.co/spaces/{CONFIG['hf_space_repo']}", flush=True)
os.makedirs(os.path.join(CONFIG["plots_dir"]), exist_ok=True)
out_json = os.path.join(CONFIG["plots_dir"], "results_summary.json")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump({k: {"baseline": float(RESULTS[k][0]), "trained": float(RESULTS[k][1])} for k in RESULTS}, f, indent=2)
print("Saved", out_json, flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 16: Push HF Space + GitHub ───────────────────────────────────────
import os
import shutil
import subprocess
from huggingface_hub import HfApi

_tok = globals().get("HF_TOKEN") or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN", "")
api = HfApi(token=_tok or None)
if CONFIG.get("push_to_hub") and _tok:
    print("=== Pushing to HuggingFace Space ===", CONFIG["hf_space_repo"], flush=True)
    try:
        api.upload_folder(
            folder_path=CONFIG["plots_dir"],
            path_in_repo="plots",
            repo_id=CONFIG["hf_space_repo"],
            repo_type="space",
            commit_message="Add training plots (4 PNG + artifacts)",
        )
    except Exception as e0:
        print("  plots", e0, flush=True)
    odir = CONFIG["output_dir"]
    if os.path.isdir(odir) and len(os.listdir(odir)) > 0:
        try:
            api.upload_folder(
                folder_path=odir, path_in_repo="checkpoints", repo_id=CONFIG["hf_space_repo"], repo_type="space", commit_message="GRPO checkpoints"
            )
        except Exception as e1:
            print("  ckpt", e1, flush=True)
    for s_n, t_n in [("before_after_examples.txt", "plots/before_after_examples.txt")]:
        p0 = os.path.join(CONFIG["plots_dir"], s_n)
        if os.path.isfile(p0):
            try:
                api.upload_file(
                    path_or_fileobj=p0, path_in_repo=t_n, repo_id=CONFIG["hf_space_repo"], repo_type="space", commit_message="qualitative"
                )
            except Exception as e2:
                print("  ", s_n, e2, flush=True)
    nbp = "/content/train_vitalchain.ipynb"
    if os.path.isfile(nbp):
        try:
            api.upload_file(
                path_or_fileobj=nbp, path_in_repo="train_vitalchain.ipynb", repo_id=CONFIG["hf_space_repo"], repo_type="space", commit_message="notebook"
            )
        except Exception as e3:
            print("  nb", e3, flush=True)
    print("HF: https://huggingface.co/spaces/%s" % (CONFIG["hf_space_repo"],), flush=True)
else:
    print("push skipped: need push_to_hub + HF_TOKEN", flush=True)
R = "/content/VitalChain"
if os.path.isdir(R):
    for f in (
        "reward_curve.png", "reward_components.png", "violation_rate.png", "cooperation_and_equity.png",
        "before_after_examples.txt", "results_summary.json", "readme_results_section.md",
    ):
        a, b = os.path.join(CONFIG["plots_dir"], f), os.path.join(R, "plots", f)
        os.makedirs(os.path.dirname(b), exist_ok=True)
        if os.path.exists(a):
            try:
                shutil.copy2(a, b)
            except Exception as e4:
                print("cp", f, e4, flush=True)
    if os.path.isfile("/content/train_vitalchain.ipynb"):
        shutil.copy2("/content/train_vitalchain.ipynb", os.path.join(R, "train_vitalchain.ipynb"))
    subprocess.run(
        "git -C " + R + " add plots/ train_vitalchain.ipynb 2>/dev/null; " + f'git -C {R} commit -m "hackathon assets" 2>/dev/null; git -C ' + R + " push 2>&1",
        shell=True, capture_output=True, text=True,
    )
    print("Git: https://github.com/singhhrishabh/VitalChain (push attempted)\n", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 17: Auto-patch README + push ────────────────────────────────────
import os
import json
import numpy as np
import subprocess
from huggingface_hub import HfApi, login

Rpath = "/content/VitalChain/README.md"
js = os.path.join(CONFIG["plots_dir"], "results_summary.json")
D = json.load(open(js, "r", encoding="utf-8")) if os.path.isfile(js) else {}
blr = D.get("Avg Episode Reward", {}).get("baseline", 0) or 0.0
trr = D.get("Avg Episode Reward", {}).get("trained", 0) or 0.0
im = 100.0 * (trr - blr) / (abs(blr) + 1e-9) if blr else 0.0
b_bt, t_bt = D.get("Blood Type Violations %", {"baseline": 0, "trained": 0}).get("baseline", 0) or 0, D.get("Blood Type Violations %", {}).get("trained", 0) or 0
b_ia, t_ia = D.get("Inaction Violations %", {"baseline": 0, "trained": 0}).get("baseline", 0) or 0, D.get("Inaction Violations %", {}).get("trained", 0) or 0
b_eq, t_eq = D.get("Equity Score (0-1)", {}).get("baseline", 0) or 0, D.get("Equity Score (0-1)", {}).get("trained", 0) or 0
b_tr, t_tr = D.get("Transport Efficiency", {}).get("baseline", 0) or 0, D.get("Transport Efficiency", {}).get("trained", 0) or 0
wu = WANDB_URL if "WANDB_URL" in dir() else (wandb.run.get_url() if wandb.run is not None else "https://wandb.ai/")

def _patch_readme() -> str:
    return f"""## 📊 Results: What Changed After Training

### Training run: GRPO, {CONFIG['model_name']}, {CONFIG.get('eval_episodes', 50)} eval ep

| Metric | Random Baseline | Trained | Δ |
|--------|-----------------|---------|---|
| Avg Episode Reward | {blr:.2f} | {trr:.2f} | {im:+.0f}% |
| Blood Type Violations | {b_bt:.1f}% | {t_bt:.1f}% | {t_bt - b_bt:+.1f}pp |
| Inaction Violations | {b_ia:.1f}% | {t_ia:.1f}% | {t_ia - b_ia:+.1f}pp |
| Equity | {b_eq:.3f} | {t_eq:.3f} | {t_eq - b_eq:+.3f} |
| Transport | {b_tr:.3f} | {t_tr:.3f} | {t_tr - b_tr:+.3f} |

### Curves
![r](plots/reward_curve.png)
### Components
![c](plots/reward_components.png)
### Violations
![v](plots/violation_rate.png)
### Cooperation
![c2](plots/cooperation_and_equity.png)

[before/after](plots/before_after_examples.txt) · [W&B]({wu}) · [Space](https://huggingface.co/spaces/{CONFIG['hf_space_repo']})
"""

sec = _patch_readme()
open(os.path.join(CONFIG["plots_dir"], "readme_results_section.md"), "w", encoding="utf-8").write(sec)
print("readme_results_section.md written. Preview:\n", sec[:800], "\n", flush=True)
if os.path.isfile(Rpath):
    t = open(Rpath, "r", encoding="utf-8", errors="replace").read()
    s = t.find("## 📊 Results: What Changed After Training")
    if s < 0:
        t2 = t.rstrip() + "\n\n" + sec + "\n"
    else:
        e1, e2 = t.find("\n---\n", s + 5), t.find("\n## ", s + 5)
        if e1 > 0 and (e2 < 0 or e1 < e2):
            t2 = t[:s] + sec + "\n\n" + t[e1 + 5 :]
        elif e2 > 0:
            t2 = t[:s] + sec + "\n\n" + t[e2 + 1 :]
        else:
            t2 = t[:s] + sec
    open(Rpath, "w", encoding="utf-8").write(t2)
_tok2 = os.environ.get("HF_TOKEN", "")
if _tok2:
    try:
        login(token=_tok2, add_to_git_credential=True)
    except Exception:
        pass
try:
    if os.path.isfile(Rpath):
        HfApi(token=_tok2 or None).upload_file(
            Rpath, "README.md", repo_id=CONFIG["hf_space_repo"], repo_type="space", commit_message="README results"
        )
except Exception as e0:
    print("HF README upload", e0, flush=True)
if os.path.isdir("/content/VitalChain"):
    subprocess.run("git -C /content/VitalChain add README.md plots/ 2>/dev/null; git -C /content/VitalChain commit -m readme 2>/dev/null; git -C /content/VitalChain push 2>/dev/null", shell=True)
print("README update attempted.", flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 18: 2-min demo script ───────────────────────────────────────────
import numpy as np

impr_ = float(globals().get("improvement", 0.0) or 0.0)
n_tr = int(CONFIG.get("training_episodes", 300) or 300)
b_bt_ = 100.0 * float(np.mean(baseline_violations.get("blood_type", [0]) or [0]) or 0.0) if "baseline_violations" in dir() else 0.0
t_bt_ = 100.0 * float(np.mean(trained_violations.get("blood_type", [0]) or [0]) or 0.0) if "trained_violations" in dir() else 0.0
script = f"""
╔══════════════════════════════════════════════════════════════╗
║     VITALCHAIN — 2-MINUTE DEMO VIDEO SCRIPT                 ║
╚══════════════════════════════════════════════════════════════╝

[0:00–0:15]  HOOK
"Every 10 minutes in India, a patient dies waiting for an organ that
 exists somewhere in the country — not because the organ isn't there,
 but because the coordination system is broken. We built VitalChain."

[0:15–0:35]  THE PROBLEM
"VitalChain is an OpenEnv RL environment that trains an LLM to
 allocate organs, blood, and bone marrow across a 3-hospital network
 under real constraints: blood-type matching, ischemic time windows,
 Green Corridor routing, partial observability, and equity."
(Show: HF Space → https://huggingface.co/spaces/singhhrishabhh/VitalChain)

[0:35–1:00]  LIVE ENVIRONMENT
"Here's the environment. The agent sees this observation with DYING patients, inventory, and transport options."
(Show: observation in Space or this notebook)
"An untrained policy can pick a suboptimal action; the trained one improves mean return."

[1:00–1:25]  TRAINING RESULTS
"After {n_tr} target training episodes in CONFIG, GRPO refines the policy;
 mean return improves ~{impr_:+.0f}% vs random baseline in our eval; blood-type
 proxy episodes: {b_bt_:.0f}% → {t_bt_:.1f}% (qualitative; see violation plot)."
(Show: reward_curve.png, then violation_rate.png briefly)

[1:25–1:50]  WHY IT MATTERS
"Multi-constraint resource allocation under time pressure with equity."
[1:50–2:00]  CLOSE
"VitalChain — the TCP/IP for biological logistics."
  → https://huggingface.co/spaces/singhhrishabhh/VitalChain
"""
print(script, flush=True)


In [ ]:
# VITALCHAIN HACKATHON 2026
# ── Cell 19: Checklist + wandb.finish ───────────────────────────────────
import os
import numpy as np
import wandb

R = "/content/VitalChain" if os.path.isdir("/content/VitalChain") else "."
Pdir = (os.path.join(R, "plots") if os.path.isdir(os.path.join(R, "plots")) else CONFIG["plots_dir"])
CHECKS = {
    "plots/reward_curve in repo or local": os.path.exists(os.path.join(Pdir, "reward_curve.png")) or os.path.exists(os.path.join(CONFIG["plots_dir"], "reward_curve.png")),
    "all 4 PNG in plots_dir": all(os.path.exists(os.path.join(CONFIG["plots_dir"], f)) for f in (
        "reward_curve.png", "reward_components.png", "violation_rate.png", "cooperation_and_equity.png",
    )),
    "before_after in plots": os.path.exists(os.path.join(CONFIG["plots_dir"], "before_after_examples.txt")),
    "train_vitalchain.ipynb": os.path.exists("/content/train_vitalchain.ipynb") or os.path.exists("train_vitalchain.ipynb"),
    "checkpoints (optional)": (not os.path.isdir(CONFIG["output_dir"])) or len(os.listdir(CONFIG["output_dir"] or ".") or []) > 0,
    "baseline_rewards.npy": os.path.exists("baseline_rewards.npy"),
    "trained_rewards.npy": os.path.exists("trained_rewards.npy"),
    "results_summary.json": os.path.exists(os.path.join(CONFIG["plots_dir"], "results_summary.json")),
    "W&B": wandb.run is not None,
}
print("\n" + "=" * 68, "\n  VITALCHAIN — CHECKLIST\n", "=" * 68, sep="")
ap = True
for c, p in CHECKS.items():
    print("  [", ("PASS" if p else "FAIL"), "]  ", c, sep="")
    if not p:
        ap = False
print("-" * 68)
if ap:
    print("\n  ALL CHECKS PASSED (or optional skipped).")
    print("  SUBMIT: https://huggingface.co/spaces/%s" % (CONFIG["hf_space_repo"],))
    if "WANDB_URL" in dir() and WANDB_URL:
        print("  W&B: ", WANDB_URL)
    print("  GitHub: https://github.com/singhhrishabh/VitalChain", flush=True)
else:
    print("  Some FAIL — fix or re-run.", flush=True)
print("=" * 68, "\n", flush=True)
try:
    wandb.finish()
except Exception as e0:
    print("wandb.finish", e0, flush=True)
print("Pipeline end.", flush=True)
